# Provisional weekly connection behavior
Offline analysis of the saved profiling cache. No API calls. This proposes a descriptive metric, not a drive-failure definition. Week labels use **device wall-calendar dates provisionally** because source offsets are absent. Do not claim a confirmed reporting timezone or a causal release regression.

The analysis uses `data/analysis_snapshot_20261027/`: an offline reconstruction of the complete backup plus completed refresh. The active profiling cache was subsequently incomplete. Source and reconstructed file checksums are recorded in the local snapshot manifest; active cache files are not overwritten.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dbt" / "dbt_project.yml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook within the project directory.")
CACHE = ROOT / "data/analysis_snapshot_20261027"
scope = json.loads((CACHE / "scope.json").read_text())
as_of = pd.Timestamp(scope["as_of"])
raw = {}
for name in ["drives", "warnings"]:
    state = json.loads((CACHE / f"{name}.json").read_text())
    if not state["complete"]:
        raise RuntimeError(f"{name}: finish extraction first")
    frame = pd.json_normalize(state["records"])
    receipt = pd.to_datetime(frame["received_at"], utc=True, errors="coerce")
    raw[name] = frame.loc[receipt.notna() & receipt.le(as_of)].copy()
print("Analysis receipt cutoff:", as_of)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 1. Verify business identity before collapsing replays
Compare every observed field except `received_at`. Exclude IDs with business-field conflicts instead of choosing a winner. For agreeing records retain the earliest receipt as availability time and latest receipt as replay metadata.

In [ ]:
clean = {}
audit = []
for name, key in [("drives", "session_id"), ("warnings", "warning_id")]:
    frame = raw[name]
    business = frame.drop(columns="received_at").drop_duplicates()
    conflicting = business.loc[business[key].notna() & business[key].duplicated(keep=False), key].unique()
    safe = frame.loc[frame[key].notna() & ~frame[key].isin(conflicting)].copy()
    receipts = safe.groupby(key)["received_at"].agg(first_received_at="min", last_received_at="max", report_count="size")
    clean[name] = safe.drop_duplicates(key).drop(columns="received_at").join(receipts, on=key)
    assert clean[name][key].is_unique
    audit.append({"dataset": name, "raw_rows": len(frame), "missing_id_rows": int(frame[key].isna().sum()), "business_conflicting_ids": len(conflicting), "canonical_records": len(clean[name]), "collapsed_replay_rows": len(safe)-len(clean[name])})
audit = pd.DataFrame(audit)
display(audit)
d = clean["drives"].copy()
w = clean["warnings"].copy()

## 2. Timestamp evidence
Device fields are parsed without attaching a timezone. Device wall time minus UTC receipt clock is an offset/lag diagnostic, **not latency**. Country is insufficient to infer a device timezone. Daylight-saving changes also prevent assuming a constant offset. Warning timestamps cannot reliably establish week boundaries.

In [ ]:
for frame, columns in [(d, ["started_at", "ended_at"]), (w, ["device_time"])]:
    for column in columns:
        if frame[column].dropna().str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True).any():
            raise RuntimeError("Unexpected timezone-bearing device value; review parsing")
        frame[column] = pd.to_datetime(frame[column], format="mixed", errors="coerce")
d["first_receipt_utc"] = pd.to_datetime(d["first_received_at"], utc=True)
d["wall_minus_receipt_hours"] = (d["ended_at"] - d["first_receipt_utc"].dt.tz_localize(None)).dt.total_seconds()/3600
display(d.groupby("country", dropna=False)["wall_minus_receipt_hours"].agg(["count", "median", "min", "max"]).round(2))
print("Warning timestamp range:", w["device_time"].min(), "to", w["device_time"].max())
clock_future = w["device_time"].gt(as_of.tz_localize(None) + pd.Timedelta(days=2))
print("Warnings more than two calendar days beyond cutoff (clock flag):", int(clock_future.sum()))
matched = w.merge(d[["session_id", "user_id", "started_at", "ended_at"]], on="session_id", how="left", suffixes=("_warning", "_drive"), indicator=True, validate="many_to_one")
is_match = matched["_merge"].eq("both")
comparable = is_match & matched["device_time"].notna() & matched["started_at"].notna() & matched["ended_at"].notna()
outside = comparable & (matched["device_time"].lt(matched["started_at"]) | matched["device_time"].gt(matched["ended_at"]))
display(pd.Series({"warning_rows": len(w), "matched_warnings": int(is_match.sum()), "match_pct": round(100*is_match.mean(), 3), "unmatched": int((~is_match).sum()), "outside_drive_interval": int(outside.sum()), "user_id_mismatches": int((is_match & matched["user_id_warning"].notna() & matched["user_id_drive"].notna() & matched["user_id_warning"].ne(matched["user_id_drive"])).sum())}, name="value").to_frame())

## 3. Eligibility and provisional Monday–Sunday weeks
Denominator: unique reported sessions with parseable start/end, end at or after start, positive numeric duration agreeing with elapsed time within one second, and integer connection count at least one. Numerator: those eligible sessions with count > 1. No missing/invalid values are converted to zero. Deduplicated report volume is displayed separately. Missing platform/version values stay in an explicit unknown group. Weeks are start-date wall-calendar weeks; current week is excluded. Volume is still subject to late reports and sessions never reported.

In [ ]:
d["duration_s"] = pd.to_numeric(d["duration_s"], errors="coerce")
d["connection_count"] = pd.to_numeric(d["connection_count"], errors="coerce")
elapsed = (d["ended_at"]-d["started_at"]).dt.total_seconds()
d["eligible"] = (d["started_at"].notna() & d["ended_at"].notna() & elapsed.ge(0) & d["duration_s"].gt(0) & (d["duration_s"]-elapsed).abs().le(1) & d["connection_count"].ge(1) & d["connection_count"].mod(1).eq(0))
d["multiple_connections"] = d["connection_count"].gt(1)
d["week"] = d["started_at"].dt.to_period("W-SUN").dt.start_time
today = as_of.tz_convert("Europe/Copenhagen").tz_localize(None).normalize()
this_monday = today-pd.Timedelta(days=today.dayofweek)
last_week = this_monday-pd.Timedelta(days=7)
previous_week = this_monday-pd.Timedelta(days=14)
print("Provisional comparison weeks:", previous_week.date(), "and", last_week.date())
display(pd.Series({"canonical_reported_drives": len(d), "eligible_drives": int(d["eligible"].sum()), "excluded_from_metric": int((~d["eligible"]).sum())}, name="rows").to_frame())
cohort = d.loc[d["eligible"] & d["week"].isin([previous_week, last_week])].copy()
for col in ["platform", "app_version", "device"]:
    cohort[col] = cohort[col].fillna("unknown")

def summarize(frame, keys):
    out = frame.groupby(keys, dropna=False).agg(eligible_drives=("session_id", "size"), drivers=("user_id", "nunique"), drives_with_multiple_connections=("multiple_connections", "sum"), median_duration_s=("duration_s", "median"), p95_connections=("connection_count", lambda s:s.quantile(.95)))
    out["multiple_connection_pct"] = 100*out["drives_with_multiple_connections"]/out["eligible_drives"]
    involved = frame.loc[frame["multiple_connections"]].groupby(keys, dropna=False)["user_id"].nunique()
    out["drivers_with_multiple_connection_drives"] = involved.reindex(out.index, fill_value=0)
    return out.round(3)
weekly = summarize(cohort, ["week"])
platforms = summarize(cohort, ["week", "platform"])
releases = summarize(cohort, ["week", "platform", "app_version"])
display(weekly)
display(platforms)
display(releases)
# Separate volume reconciliation: all canonical reports versus metric denominator.
display(d.loc[d["week"].isin([previous_week,last_week])].groupby("week").agg(reported_drives=("session_id","size"), eligible_drives=("eligible","sum")))
assert cohort["session_id"].is_unique
assert weekly["eligible_drives"].sum() == len(cohort)
assert weekly["drives_with_multiple_connections"].le(weekly["eligible_drives"]).all()
assert weekly["multiple_connection_pct"].between(0,100).all()

## 4. Mix and arrival diagnostics
Duration strata and device groups expose obvious differences in population mix; they do not remove confounding. Receipt timing relative to the following Monday in Copenhagen is shown as reporting availability, not event-to-ingestion lag. The first receipt is the earliest *observed* receipt. Reports can still arrive after the current cutoff.

In [ ]:
cohort["duration_band"] = pd.cut(cohort["duration_s"], bins=[0,600,1800,float("inf")], labels=["under_10min", "10_to_30min", "over_30min"], right=False)
display(summarize(cohort, ["week", "platform", "duration_band"]).dropna(how="all"))
display(summarize(cohort, ["week", "platform", "device"]))
following_monday = (cohort["week"]+pd.Timedelta(days=7)).dt.tz_localize("Europe/Copenhagen")
cohort["received_after_following_monday"] = cohort["first_receipt_utc"].ge(following_monday)
display(cohort.groupby("week").agg(drives=("session_id","size"), received_after_following_monday=("received_after_following_monday","sum")))
# Sensitivity only: shifting all start clocks by +/-2 hours can move boundary sessions.
checks=[]
for shift in [-2,0,2]:
    shifted=(d["started_at"]+pd.Timedelta(hours=shift)).dt.to_period("W-SUN").dt.start_time
    for week in [previous_week,last_week]:
        sample=d.loc[d["eligible"] & shifted.eq(week)]
        checks.append({"uniform_clock_shift_hours":shift,"week":week,"drives":len(sample),"multiple_connection_pct":round(100*sample["multiple_connections"].mean(),3)})
display(pd.DataFrame(checks))
print("Uniform clock shifts are sensitivity checks, not validated timezone conversions.")

## Focused iOS 6.3.0 investigation
Use canonical, eligible iOS sessions. Show all-history distributions, then compare the same two complete weeks to reduce calendar confounding. Earlier versions still have different users and rollout populations. Warning counts below include only deduplicated warnings matched to the session; interval-valid activity excludes warning timestamps outside the drive. Zero observed warnings is not proof that an alert was missed. `device` identifies CO-DRIVER categories (CD1/CD2/CD3), not iPhone models; handset models are unavailable.

In [ ]:
ios = d.loc[d["eligible"] & d["platform"].eq("ios")].copy()
warning_counts = matched.loc[is_match].groupby("session_id").size()
valid_warning_counts = matched.loc[comparable & ~outside].groupby("session_id").size()
for column, counts in [("matched_warning_count", warning_counts), ("interval_valid_warning_count", valid_warning_counts)]:
    ios[column] = ios["session_id"].map(counts).fillna(0).astype(int)
ios["device"] = ios["device"].fillna("unknown")

def ios_summary(frame, keys):
    result = frame.groupby(keys, dropna=False).agg(
        drives=("session_id","size"), drivers=("user_id","nunique"),
        multiple_connection_drives=("multiple_connections","sum"),
        connection_min=("connection_count","min"), connection_p50=("connection_count","median"),
        connection_p95=("connection_count",lambda x:x.quantile(.95)), connection_max=("connection_count","max"),
        duration_p50_s=("duration_s","median"), duration_p95_s=("duration_s",lambda x:x.quantile(.95)),
        total_duration_s=("duration_s","sum"), matched_warnings=("matched_warning_count","sum"),
        interval_valid_warnings=("interval_valid_warning_count","sum"),
        zero_observed_warning_drives=("matched_warning_count",lambda x:x.eq(0).sum()))
    result["multiple_connection_pct"] = 100*result["multiple_connection_drives"]/result["drives"]
    result["matched_warnings_per_drive"] = result["matched_warnings"]/result["drives"]
    result["valid_warnings_per_drive_hour"] = result["interval_valid_warnings"]/(result["total_duration_s"]/3600)
    result["zero_observed_warning_pct"] = 100*result["zero_observed_warning_drives"]/result["drives"]
    return result.round(3)

ios_history = ios_summary(ios,["app_version"])
ios_connections = pd.crosstab(ios["app_version"],ios["connection_count"])
ios_devices = ios_summary(ios,["app_version","device"])
ios_recent = ios.loc[ios["week"].isin([previous_week,last_week])]
ios_recent_summary = ios_summary(ios_recent,["week","app_version"])
print("All-history iOS version summary (different calendar/rollout populations)")
display(ios_history)
print("All-history connection counts: columns are connection_count values")
display(ios_connections)
print("iOS version x CO-DRIVER device category, all history")
display(ios_devices[["drives","multiple_connection_drives","connection_min","connection_max","duration_p50_s","matched_warnings_per_drive"]])
print("Same-calendar-week comparison")
display(ios_recent_summary)
assert ios["session_id"].is_unique
assert ios["interval_valid_warning_count"].le(ios["matched_warning_count"]).all()
assert ios_history["drives"].sum() == len(ios)
print("No causal inference: counter semantics, expected warnings, and drive-failure outcomes remain unobserved.")


## 5. Review outcome
These checks support a provisional weekly connection-behavior comparison. Confirm timestamp semantics and what the count represents before promoting this to a release-health KPI. No causality or drive-failure claim is made. No dbt models or dashboards are created.

In [ ]:
# Export aggregate evidence as no more than three Product findings.
def markdown_table(frame):
    table = frame.reset_index().copy()
    for col in table.columns:
        if pd.api.types.is_datetime64_any_dtype(table[col]):
            table[col] = table[col].dt.strftime("%Y-%m-%d")
    columns = [str(c) for c in table.columns]
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"]*len(columns)) + " |"]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in table.itertuples(index=False, name=None)]
    return "\n".join(lines)

v630 = ios_history.loc["6.3.0"]
change = weekly.loc[last_week,"multiple_connection_pct"]-weekly.loc[previous_week,"multiple_connection_pct"]
report = f"""# Weekly Product findings

As of {as_of.isoformat()}; cached snapshot, no new API requests.
Comparison: {previous_week.date()}–{(previous_week+pd.Timedelta(days=6)).date()}
versus {last_week.date()}–{(last_week+pd.Timedelta(days=6)).date()}.
Weeks are provisional device wall-calendar weeks, not a confirmed timezone conversion.

## 1. The aggregate connection signal improves, but cannot establish better drive experience

The share of eligible reported drives with multiple connections changed from
{weekly.loc[previous_week,'multiple_connection_pct']:.3f}% to
{weekly.loc[last_week,'multiple_connection_pct']:.3f}% ({change:.3f} percentage points).
Android moved from {platforms.loc[(previous_week,'android'),'multiple_connection_pct']:.3f}% to
{platforms.loc[(last_week,'android'),'multiple_connection_pct']:.3f}%; iOS from
{platforms.loc[(previous_week,'ios'),'multiple_connection_pct']:.3f}% to
{platforms.loc[(last_week,'ios'),'multiple_connection_pct']:.3f}%.
Changing iOS version adoption can affect the aggregate. This is recorded connection
behavior, not a failed-drive rate or evidence of a release causing improvement.

{markdown_table(weekly[['eligible_drives','drives_with_multiple_connections','multiple_connection_pct','drivers_with_multiple_connection_drives']])}

## 2. iOS 6.3.0 raises a counter/instrumentation question across device categories

Across all available eligible history, iOS 6.3.0 has {int(v630['drives']):,} drives,
{int(v630['multiple_connection_drives']):,} with multiple connections, and a connection
range of {v630['connection_min']:g}–{v630['connection_max']:g}.
The version/device table tests whether this persists for each observed category.
`device` describes CO-DRIVER categories, not iPhone models. Some earlier-version
cohorts have few drivers; history-wide comparisons also have different date coverage.
Check counter semantics or release instrumentation before treating its zero rate
as evidence of stability. These observations do not prove an instrumentation defect.

{markdown_table(ios_connections)}

{markdown_table(ios_devices[['drives','multiple_connection_drives','connection_min','connection_max']])}

## 3. Duration and warning activity provide context, not proof of missed alerts

For 6.3.0 across available history, median duration is {v630['duration_p50_s']:g} seconds,
p95 is {v630['duration_p95_s']:g} seconds, and observed matched warnings average
{v630['matched_warnings_per_drive']:g} per drive. Compare versions in the same weeks
below; warning rates depend on exposure, routes, warning types, and driver mix.
In the latest complete week, 6.3.0 and 6.1.1 have similar interval-valid warning activity ({ios_recent_summary.loc[(last_week,"6.3.0"),"valid_warnings_per_drive_hour"]:g} versus {ios_recent_summary.loc[(last_week,"6.1.1"),"valid_warnings_per_drive_hour"]:g} per drive-hour), with median durations {ios_recent_summary.loc[(last_week,"6.3.0"),"duration_p50_s"]:g} versus {ios_recent_summary.loc[(last_week,"6.1.1"),"duration_p50_s"]:g} seconds. There is no clear warning-activity collapse in this comparison. Neither a lower warning rate nor a zero-warning drive proves an alert was missed.
There is no expected-warning denominator or explicit drive-failure outcome.

{markdown_table(ios_recent_summary[['drives','drivers','multiple_connection_pct','duration_p50_s','duration_p95_s','matched_warnings_per_drive','valid_warnings_per_drive_hour','zero_observed_warning_pct']])}

Quality and scope: receipt-only replays are collapsed before analysis. The canonical
warning join matches {100*is_match.mean():.3f}%; {int((~is_match).sum())} warnings are unmatched.
{int(outside.sum())} matched warnings are outside drive intervals and excluded from
interval-valid warning rates. Zero observed warning counts can reflect incomplete
source reporting. Full API pagination does not establish complete attempted drives.
Late reports can revise weekly results; historical backdated receipts outside the
refresh overlap can be missed. Driver counts are not additive across versions.
See [metric_definitions.md](metric_definitions.md) for eligibility and exclusions.
"""
(ROOT / "reports").mkdir(exist_ok=True)
(ROOT / "reports/weekly_findings.md").write_text(report)
print("Saved three Product findings to reports/weekly_findings.md.")
